# DDPM Face Generation — FFHQ 100 Images

**Objective:** Train a simplified Denoising Diffusion Probabilistic Model (DDPM) on 100 FFHQ face images resized to 64×64, then generate images from random noise.

> This is an educational prototype. With only 100 images and a small U-Net, outputs may remain noisy or blurry. Training loss alone does not guarantee realistic samples.

## 1. Imports and device

In [ ]:
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Dataset setup

Upload `100 images.zip` to Google Drive once, under `MyDrive/DDPM_Project/`. This cell mounts Drive and extracts only if needed. It avoids `files.upload()`, which asks for the ZIP again after a runtime restart.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ZIP_PATH = Path("/content/drive/MyDrive/DDPM_Project/100 images.zip")
RAW_DIR = Path("/content/dataset")
RAW_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"ZIP not found: {ZIP_PATH}. Upload it to the Drive folder above.")

# Extract only if no images are already present
extensions = {".jpg", ".jpeg", ".png", ".webp"}
existing = [p for p in RAW_DIR.rglob("*") if p.is_file() and p.suffix.lower() in extensions]
if not existing:
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(RAW_DIR)

image_paths = sorted(
    p for p in RAW_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
)
print("Images found:", len(image_paths))
if not image_paths:
    raise RuntimeError("No images found. Check ZIP contents and RAW_DIR.")

## 3. Preprocess images to 64×64 RGB

In [ ]:
IMAGE_DIR = Path("/content/ffhq_64")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# Rebuild the processed folder to avoid stale/duplicate files
for p in IMAGE_DIR.glob("*.png"):
    p.unlink()

for i, path in enumerate(image_paths):
    with Image.open(path) as img:
        img = img.convert("RGB").resize((64, 64), Image.Resampling.LANCZOS)
        img.save(IMAGE_DIR / f"face_{i:03d}.png")

print("Processed images:", len(list(IMAGE_DIR.glob("*.png"))))

## 4. Dataset and DataLoader
`ToTensor()` converts pixels to `[0,1]`; `Normalize(0.5, 0.5)` converts them to `[-1,1]`.

In [ ]:
class FFHQDataset(Dataset):
    def __init__(self, folder):
        self.paths = sorted(Path(folder).glob("*.png"))
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,)*3, (0.5,)*3)
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as img:
            return self.transform(img.convert("RGB"))

dataset = FFHQDataset(IMAGE_DIR)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, drop_last=False)

batch = next(iter(dataloader))
print("Dataset size:", len(dataset))
print("Batch shape:", batch.shape)
print("Pixel range:", batch.min().item(), "to", batch.max().item())

## 5. Forward diffusion schedule
We define `beta`, `alpha = 1-beta`, and cumulative `alpha_bar`. `q_sample` adds noise at a selected timestep.

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def q_sample(x0, t):
    noise = torch.randn_like(x0)
    alpha_bar_t = alpha_bars[t].view(-1, 1, 1, 1)
    xt = torch.sqrt(alpha_bar_t) * x0 + torch.sqrt(1 - alpha_bar_t) * noise
    return xt, noise

# Quick shape check
x0 = next(iter(dataloader)).to(device)
t = torch.randint(0, T, (x0.shape[0],), device=device)
xt, noise = q_sample(x0, t)
print("x0:", x0.shape, "| xt:", xt.shape, "| noise:", noise.shape)

## 6. U-Net noise predictor
This compact U-Net has one downsampling stage, timestep embedding, one skip connection, and predicts RGB-shaped noise.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, T=1000):
        super().__init__()
        self.encoder = ConvBlock(3, 32)
        self.downsample = nn.Conv2d(32, 64, 3, stride=2, padding=1)

        self.time_embedding = nn.Embedding(T, 32)
        self.time_projection = nn.Linear(32, 64)

        self.upsample = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.decoder = ConvBlock(64, 32)
        self.final_conv = nn.Conv2d(32, 3, 3, padding=1)

    def forward(self, x, t):
        skip = self.encoder(x)
        x = self.downsample(skip)

        t_emb = self.time_projection(self.time_embedding(t))
        x = x + t_emb[:, :, None, None]

        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)
        x = self.decoder(x)
        return self.final_conv(x)


model = UNet(T=T).to(device)
test_x = torch.randn(2, 3, 64, 64, device=device)
test_t = torch.randint(0, T, (2,), device=device)
print("Predicted noise shape:", model(test_x, test_t).shape)

## 7. Training setup
The model learns to predict the exact Gaussian noise added to each image. MSE is minimized with Adam.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x0 in loader:
        x0 = x0.to(device)
        t = torch.randint(0, T, (x0.shape[0],), device=device)
        xt, actual_noise = q_sample(x0, t)

        predicted_noise = model(xt, t)
        loss = criterion(predicted_noise, actual_noise)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

## 8. Train the model

Set `NUM_EPOCHS` to the experiment you intend to run. This cell starts a **new training run**. Do not run it again accidentally if you intend to continue from a saved checkpoint.

In [ ]:
NUM_EPOCHS = 10
loss_history = []

for epoch in range(NUM_EPOCHS):
    epoch_loss = train_one_epoch(model, dataloader, optimizer, criterion)
    loss_history.append(epoch_loss)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] loss: {epoch_loss:.6f}")

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Average training MSE")
plt.title("DDPM Training Loss")
plt.grid(True)
plt.show()

## 9. Save checkpoint
Save the actual completed epoch number. The checkpoint includes model and optimizer states for later continuation.

In [ ]:
CHECKPOINT_DIR = Path("/content/drive/MyDrive/DDPM_Project/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

completed_epochs = len(loss_history)
checkpoint_path = CHECKPOINT_DIR / f"ddpm_epoch_{completed_epochs}.pth"

torch.save({
    "epoch": completed_epochs,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss_history": loss_history
}, checkpoint_path)

print("Saved:", checkpoint_path)

## 10. Reverse diffusion / image generation
At each reverse step, the U-Net estimates the noise. We use the DDPM posterior variance and add stochastic noise only when `t > 0`.

In [ ]:
# Posterior variance for DDPM sampling
alpha_bars_prev = torch.cat([
    torch.ones(1, device=device),
    alpha_bars[:-1]
])
posterior_variance = betas * (1 - alpha_bars_prev) / (1 - alpha_bars)


@torch.no_grad()
def generate_images(model, num_images=4):
    model.eval()
    x = torch.randn(num_images, 3, 64, 64, device=device)

    for step in reversed(range(T)):
        t = torch.full((num_images,), step, device=device, dtype=torch.long)

        beta_t = betas[t].view(-1, 1, 1, 1)
        alpha_t = alphas[t].view(-1, 1, 1, 1)
        alpha_bar_t = alpha_bars[t].view(-1, 1, 1, 1)

        predicted_noise = model(x, t)
        mean = (x - beta_t / torch.sqrt(1 - alpha_bar_t) * predicted_noise) / torch.sqrt(alpha_t)

        if step > 0:
            variance_t = posterior_variance[t].view(-1, 1, 1, 1)
            x = mean + torch.sqrt(variance_t) * torch.randn_like(x)
        else:
            x = mean

    return x.clamp(-1, 1)

## 11. Generate and display samples

In [ ]:
generated_images = generate_images(model, num_images=4)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    img = ((generated_images[i].cpu() + 1) / 2).clamp(0, 1)
    ax.imshow(img.permute(1, 2, 0))
    ax.axis("off")

plt.suptitle(f"Generated samples after {completed_epochs} epochs")
plt.tight_layout()
plt.show()

## 12. Results and limitations

Record your actual generated output and loss here. With 100 images, 64×64 resolution, and this simplified U-Net, generated samples may be noisy or blurry. Do not claim realistic face generation unless the results support it.

**Reproducibility note:** Save the notebook, dataset instructions, loss plot, and a few generated samples. Do not upload the dataset ZIP unless you have permission to redistribute it.